# Xeno-canto Gathering

Build a reproducible download set using XC query tags.


In [1]:
import json
import sys
import time
from pathlib import Path

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.append(str(repo_root))

from src.config import CONFIG
from src.utils.xeno_canto import get_recordings_data_for_species, download_recordings


In [2]:
DATASET_ROOT = Path(CONFIG.paths.root)
RAW_DIR = DATASET_ROOT / CONFIG.paths.raw_dir
MANIFEST_DIR = DATASET_ROOT / "manifests"

RAW_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
SPECIES_FILE = Path("../../species_list_small.json")
species_map = json.loads(SPECIES_FILE.read_text(encoding="utf-8"))
species_list = list(species_map.values())
species_list[:5]


[{'common_name': 'European Robin', 'sci_name': 'Erithacus rubecula'},
 {'common_name': 'Eurasian Blackbird', 'sci_name': 'Turdus merula'},
 {'common_name': 'Eurasian Wren', 'sci_name': 'Troglodytes troglodytes'},
 {'common_name': 'Eurasian Blue Tit', 'sci_name': 'Cyanistes caeruleus'},
 {'common_name': 'Great Tit', 'sci_name': 'Parus major'}]

In [4]:
def build_query(sci_name: str) -> str:
    tags = [
        f'sp:"{sci_name}"',
        "grp:birds",
        "area:europe",
        'q:">C"',
        'len:">2"',
        'len:"<60"',
    ]
    return " ".join(tags)

PER_PAGE = 500


## Fetch metadata


In [5]:
MAX_SPECIES = 5

recordings_by_species = {}
species_to_fetch = species_list if MAX_SPECIES is None else species_list[:MAX_SPECIES]

for entry in species_to_fetch:
    sci_name = entry["sci_name"]
    query = build_query(sci_name)
    recs = get_recordings_data_for_species(query, per_page=PER_PAGE)
    recordings_by_species[sci_name] = recs
    print(f"{sci_name}: {len(recs)} recordings")


Erithacus rubecula: 2245 recordings
Turdus merula: 2277 recordings
Troglodytes troglodytes: 1806 recordings
Cyanistes caeruleus: 1561 recordings
Parus major: 3259 recordings


## Download audio


In [6]:
MAX_PER_SPECIES = 2  

def species_dir_name(sci_name: str) -> str:
    return sci_name.lower().replace(" ", "_")

for sci_name, recs in recordings_by_species.items():
    if MAX_PER_SPECIES is not None:
        recs = recs[:MAX_PER_SPECIES]
    species_dir = RAW_DIR / species_dir_name(sci_name)
    manifest_csv = MANIFEST_DIR / f"{species_dir.name}.csv"
    entries = download_recordings(
        recs,
        species_dir,
        overwrite=False,
        manifest_csv_path=manifest_csv,
    )
    print(f"{sci_name}: downloaded {len(entries)}")


Erithacus rubecula: downloaded 2
Turdus merula: downloaded 2
Troglodytes troglodytes: downloaded 2
Cyanistes caeruleus: downloaded 2
Parus major: downloaded 2
